# lp-data 形式データセットの書き出し

`development_merged.json` を med-chest-metry-pi6 が読める lp-data 標準形式へ書き出す。
CLI（`segmentation-validation export-lpdata`）と**同じ関数を呼ぶだけ**の実行・確認用ノートブック。

**変換ロジックはここに置かない。** 正本は `src/segmentation_validation/lpdata_export/` にあり、
CLI もこのノートブックも `export_lpdata(config, options)` を呼ぶ。挙動を変えたいときは
パッケージ側を直すこと（ここにコピーを作ると、CLI と結果が食い違っても誰も気づけない）。

属性の型と意味の正典は med-chest-metry-pi6 の `dataset_template_pneumothorax.yaml`（PR #68）。

対応: [README.md](../README.md) / 参照: [docs/dataset_format.md](../docs/dataset_format.md)

## 0. 前提

- カーネルはこのリポジトリの `uv` 環境（`.venv`）に紐づいていること。
  紐づいていない場合は一度ターミナルで以下を実行してからカーネルを選び直す。

  ```bash
  cd /mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation
  uv sync --group notebook
  uv run python -m ipykernel install --user --name segmentation-validation
  ```

- 入力の統合JSONは `build-dataset` が作る（README 3章）。無ければ先にそちらを回す。

In [ ]:
import os

PROJECT_ROOT = "/mnt/project/chest/metry/pi6/work/nakamura/segmentation-validation"
os.chdir(PROJECT_ROOT)
print("cwd:", os.getcwd())

## 1. 設定

**ここだけ書き換えれば以降の全セルに反映される。**

| 変数 | 意味 |
| --- | --- |
| `IMAGE_MODE` | `convert`=DICOMを16bit PNGへ変換して実ファイルを参照 / `planned`=予定パスだけ記録 / `none`=`image_file` を null |
| `MASK_MODE` | `generate`=結合気胸マスクPNGを書き出す / `planned`=予定パスだけ記録 / `none`=`pixel_array` を null |
| `NO_MEASURE` | `True` なら画素を読まず導出値を全て null にする（高速確認モード。**学習には使えない**） |
| `LIMIT` | 先頭N件だけ処理する。`None` で全件（42,574画像） |

初回は `NO_MEASURE=True` / `LIMIT` 小さめ / `IMAGE_MODE="none"` で通し、
summary を見てから重い設定へ上げるのが安全。

In [ ]:
from pathlib import Path

from segmentation_validation.config import Config
from segmentation_validation.lpdata_export import ExportOptions

config = Config()

# --- 入力 ---
MERGED_JSON_PATH = "output/development/20260914_merged/development_merged.json"
TEMPLATE_PATH = str(config.resolve(config.lpdata_export.template_path))

# --- 出力先（3つとも独立に指定できる）---
JSON_OUTPUT_PATH = "output/lpdata/20260914_merged/chest_metry_pi6_pneumothorax.json"
IMAGE_OUTPUT_DIR = "output/lpdata/20260914_merged/images"
MASK_OUTPUT_DIR = "output/lpdata/20260914_merged/masks/pneumothorax"

# --- meta の必須項目（テンプレートのダミー値を上書きする）---
DATASET_NAME = "chest_metry_pi6_pneumothorax"
DATASET_ID = "chest_metry_pi6_pneumothorax_2026_001"
OWNER = "kosuke.nakamura"

# --- モードと実行制御 ---
IMAGE_MODE = "none"       # convert / planned / none
MASK_MODE = "none"        # generate / planned / none
NO_MEASURE = True         # True: 画素を読まない（高速確認モード）
JOBS = 8
LIMIT = 200               # None で全件
ONLY_DATASETS = ()        # 例: ("PTR_CX_MT_PI3_pneumothorax",)
FORCE = False             # 計測キャッシュを捨て、既存PNGも作り直す


def build_options(**overrides) -> ExportOptions:
    """上の設定から ExportOptions を作る。個別に上書きもできる。"""
    defaults = dict(
        merged_path=Path(MERGED_JSON_PATH).absolute(),
        template_path=Path(TEMPLATE_PATH),
        out_path=Path(JSON_OUTPUT_PATH).absolute(),
        image_output_dir=Path(IMAGE_OUTPUT_DIR).absolute(),
        mask_output_dir=Path(MASK_OUTPUT_DIR).absolute(),
        dataset_name=DATASET_NAME,
        dataset_id=DATASET_ID,
        owner=OWNER,
        image_mode=IMAGE_MODE,
        mask_mode=MASK_MODE,
        measure=not NO_MEASURE,
        jobs=JOBS,
        limit=LIMIT,
        only_datasets=tuple(ONLY_DATASETS),
        force=FORCE,
    )
    return ExportOptions(**(defaults | overrides))


options = build_options()
print("入力  :", options.merged_path)
print("正典  :", options.template_path)
print("出力  :", options.out_path)
print("画像  :", options.image_mode, "->", options.image_output_dir)
print("マスク:", options.mask_mode, "->", options.mask_output_dir)
print("計測  :", "する" if options.measure else "しない（下見）")

## 2. 入力の確認

統合JSONを開いて件数とデータセットの内訳を見る。**画素は読まない**ので数十秒で終わる。

In [ ]:
import collections

from segmentation_validation.adapters.engineer_set import EngineerSetAdapter

adapter = EngineerSetAdapter(source_path=options.merged_path, config=config)
groups = list(adapter.iter_files())

institutions = {g.institution for g in groups}
patients = {(g.institution, g.patient_id) for g in groups}
print(f"画像 {len(groups):,} / 施設 {len(institutions):,} / 患者 {len(patients):,}")
print("fingerprint:", adapter.meta_development.get("fingerprint"))

by_dataset = collections.Counter(g.dataset_id for g in groups)
with_annotation = collections.Counter(g.dataset_id for g in groups if g.records)
print()
print(f"{'dataset_id':48s} {'画像':>9s} {'annotation有':>13s}")
for dataset_id, total in sorted(by_dataset.items()):
    print(f"{dataset_id:48s} {total:9,} {with_annotation[dataset_id]:13,}")

## 3. ラベル規則の確認

ラベルは**既存 annotation と明示的な正常情報だけ**から作る（読影レポートは使わない）。

- `finding_labels` … `finding_code_systems` の code_system を持つ geometry annotation の `code_text_eng`
- `abnormal_finding_status` … 所見あり→`present` / 明示の正常あり→`absent` / それ以外→`unknown`

**annotation が無いことだけを理由に `absent` にはしない。**
どの code_system を所見として採るか、どれを「明確な正常」と認めるかは
`config.lpdata_export` の2つの allowlist で決まる。下のセルで実データの分布と
突き合わせ、根拠が確認できたものだけを足していく。

In [ ]:
from segmentation_validation.lpdata_export.labels import qualified

findings = config.lpdata_export.finding_code_systems
evidence = config.lpdata_export.normal_evidence
print("所見として採る code_system     :", findings)
print("「明確な正常」と認めるラベル   :", evidence)

geometry_systems = collections.Counter(
    label.code_system for g in groups for r in g.records for label in r.labels
)
print()
print("--- geometry annotation の code_system（所見の候補）---")
for name, count in geometry_systems.most_common():
    print(f"  {'採用' if name in findings else '  — '} {name:24s} {count:9,}")

normal_like = collections.Counter(
    qualified(label)
    for g in groups
    for label in list(g.case_labels) + [x for r in g.records for x in r.labels]
    if label.code_text_eng == "normal"
)
print()
print("--- 「正常」を名乗るラベル（absent 判定の候補）---")
for name, count in normal_like.most_common():
    print(f"  {'採用' if name in evidence else '保留'} {name:24s} {count:9,}")

## 4. エクスポート実行

`export_lpdata` が CLI と同じ処理を行う。上の設定のまま実行する。

In [ ]:
import logging

from segmentation_validation.lpdata_export import export_lpdata

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

result = export_lpdata(config, options)
print()
print("サンプル数  :", f"{result.samples:,}")
print("ラベル分布  :", result.status_counts)
print("不変条件違反:", len(result.violations))

## 5. DICOM → 16bit PNG 変換

> ⚠️ **全件で uint16 生データ 251 GiB を NFS 越しに読み、PNG を約 113 GiB 書く。**
> 空き容量（`df -h /mnt/project`）と他ジョブの状況を確認してから回すこと。
> **まず `LIMIT` を小さくして所要時間を測る。**

既存のPNGは skip するので、途中で止めても続きから再開できる（作り直したいときは `FORCE=True`）。
DICOM が実在しない画像は `image_file: null` にして続行する（**空画像は作らない**）。

> 出力JSONは実行のたびに上書きされる。一時的な上書きではなく**設定そのものを進める**こと
> （そうしないと、次のセルを流したときに `image_file` が null に戻る）。

In [ ]:
# 設定を書き換えてから流す。出力JSONは毎回上書きされるので、
# 一時的な上書き（build_options(image_mode=...)）にすると、後のセルを流したときに
# image_file が null に戻ってしまう。設定そのものを進めるのが正しい。
IMAGE_MODE = "convert"

result = export_lpdata(config, build_options())
print("変換の内訳:", result.image_counts)

## 6. 結合気胸マスクの生成（必要なら）

1画像に複数の気胸 annotation があるものは OR 合成して1枚にする（実測113画像）。
面積は**必ず合成後に数える**（各マスクの面積を足すと重なり分が二重計上になる）。
全ゼロになったマスクは実体を置かず `pixel_array: null` にする。

`MASK_MODE="planned"` でも面積は計算され、`mask_merge_manifest.json` に
「どの元マスクをORしたか」と「期待される画素数」が残るので、後から別工程で作っても検算できる。

> 出力JSONは実行のたびに上書きされる。このセルは前セルの `IMAGE_MODE="convert"` を
> 引き継いだうえで `MASK_MODE` を進めるので、最終的な1本に両方が反映される。

In [ ]:
# ここも設定そのものを進める（前セルの IMAGE_MODE="convert" を引き継ぐ）。
# マスクを書き出すには画素を読む必要があるので NO_MEASURE も下ろす。
MASK_MODE = "generate"
NO_MEASURE = False

result = export_lpdata(config, build_options())
print("マスクの内訳:", result.mask_counts)
print("manifest    :", result.artifacts.get("mask_merge_manifest"))

## 7. summary を読む

`lpdata_export_summary.md` には、**次に人間が何を決めるべきか**が出る
（allowlist に入っていない正常らしきラベル、気胸データセット名なのに annotation が0件のもの、所見語彙）。

In [ ]:
from IPython.display import Markdown, display

display(Markdown(Path(result.artifacts["summary"]).read_text(encoding="utf-8")))

## 8. 集計を確認する

`abnormal_finding_status` × `pneumothorax_case` のクロス集計と、属性ごとの null 率。

**`unknown` × `pneumothorax_case: true` は初期エクスポートでは 0件が正しい。**
「気胸ラベルはあるが読影所見は未取得」という組み合わせは PR #68 が正当と認めているが、
既存 annotation だけで作る限り（気胸 annotation があれば必ず `present` になるので）出ない。
後日の読影レポートCSV更新で初めて現れる。

In [ ]:
import json

from segmentation_validation.lpdata_export import check_samples, read_dataset

dataset = read_dataset(options.out_path)
samples = dataset["samples"]
print(f"サンプル数 {len(samples):,}")

print()
print("--- abnormal_finding_status × pneumothorax_case ---")
cross = collections.Counter(
    (s["abnormal_finding_status"], s["pneumothorax_case"]) for s in samples.values()
)
for (status, case), count in sorted(cross.items()):
    print(f"  {status:8s} × case={str(case):5s} {count:9,}")
print("  ↑ unknown × True は初期エクスポートでは 0件が正しい")

print()
print("--- 値が無い属性（null または []）---")
def is_empty(value):
    return value is None or value == [] or value == {"pixel_array": None}

for key in dataset["meta"]["structure"]:
    count = sum(1 for s in samples.values() if is_empty(s[key]))
    if count:
        print(f"  {key:26s} {count:9,} ({count / len(samples):.1%})")

print()
print("--- 不変条件 ---")
violations = check_samples(samples)
print("  違反:", len(violations))
for violation in violations[:10]:
    print("   -", violation)

## 9. 出力ファイルを数件見る

`meta` と代表的なサンプル、および参照先ファイルが実在するかを確認する。

In [ ]:
print(json.dumps(
    {k: v for k, v in dataset["meta"].items() if k != "structure"},
    ensure_ascii=False, indent=2,
))

positives = [sid for sid, s in sorted(samples.items()) if s["pneumothorax_case"]][:2]
others = [sid for sid in sorted(samples) if sid not in positives][:1]
picked = positives + others

print()
print("--- サンプル例 ---")
for sid in picked:
    print(json.dumps({sid: samples[sid]}, ensure_ascii=False, indent=2))

print()
print("--- 参照先ファイルの実在確認 ---")
base = options.out_path.parent
for sid in picked:
    for key in ("dicom_file", "image_file", "pneumothorax_mask"):
        value = samples[sid][key]
        if isinstance(value, dict):
            value = value["pixel_array"]
        if value is None:
            print(f"  {sid} {key:20s} null")
            continue
        path = Path(value) if Path(value).is_absolute() else base / value
        print(f"  {sid} {key:20s} {'あり' if path.exists() else '無し'} {path}")